### Prepare the PCA dataset

Prepare the flow measurements for PCA and keep their labels and timestamps for Section 2 of the assignment.


In [ ]:
import pandas as pd
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.preprocessing import DropAllZeroColumns, SpearmanFilter

In [ ]:
df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip()
flows = df.set_index("Unnamed: 0").drop(
    columns=[
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
    ]
)

### Choose the PCA inputs

The original row ID keeps each flow aligned with its label and timestamp. `Flow ID` duplicates information in other columns. The prepared dataset also omits the source and destination IP addresses and ports.

PCA uses measurements and counts. `Inbound` and protocol indicators such as `Protocol_UDP` represent categories and stay with the metadata for inspection.


In [ ]:
# Protocol numbers from the IANA registry.
# https://www.iana.org/assignments/protocol-numbers/protocol-numbers.xhtml
protocol_names = {0: "HOPOPT", 6: "TCP", 17: "UDP"}
flows["Protocol"] = (
    flows["Protocol"]
    .map(protocol_names)
    .fillna("Unknown_" + flows["Protocol"].astype(str))
)
flows = pd.get_dummies(flows, columns=["Protocol"], dtype=int)

categorical_columns = {"label", "Inbound"}
measurement_columns = [
    column
    for column in flows.select_dtypes(include="number").columns
    if column not in categorical_columns and not column.startswith("Protocol_")
]
X = flows[measurement_columns]
metadata = flows.drop(columns=measurement_columns)
y = metadata["label"].copy()

### Define the preprocessing pipeline

1. Drop measurements that are zero in every row used to fit the pipeline.
2. Scan the remaining measurements in column order. Drop a measurement if its absolute Spearman correlation with any retained measurement exceeds 0.9. For example, if two columns have a correlation of -0.95, keep the first.
3. Standardize the retained measurements so their units do not dominate PCA.
4. Keep enough principal components to explain at least 99% of the variance in the standardized measurements.
5. Name the component columns `PC1`, `PC2`, and so on. Pandas output carries the original row IDs through scaling and PCA.


In [ ]:
def name_components(scores):
    names = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
    return scores.set_axis(names, axis="columns")


preprocess = Pipeline(
    [
        ("drop_all_zero", DropAllZeroColumns()),
        ("spearman", SpearmanFilter(threshold=0.9)),
        ("scale", StandardScaler().set_output(transform="pandas")),
        (
            "pca",
            PCA(n_components=0.99, svd_solver="full").set_output(transform="pandas"),
        ),
        ("name_components", FunctionTransformer(name_components)),
    ]
)

### Inspect the exploratory fit

Fit on all flows to inspect the feature removals and retained variance. `X_reduced` contains the components. `df_pca` joins them to the label and other metadata by row ID for analysis.


In [ ]:
X_reduced = preprocess.fit_transform(X)
df_pca = pd.concat([X_reduced, metadata], axis=1)

In [ ]:
columns_to_drop = preprocess.named_steps["drop_all_zero"].dropped_columns_
correlation_removals = preprocess.named_steps["spearman"].removals_
pca = preprocess.named_steps["pca"]
variance_retained = pca.explained_variance_ratio_.sum()

print(f"All-zero columns removed: {', '.join(columns_to_drop)}")
print(f"{pca.n_components_} components retain {variance_retained:.2%} of variance.")
correlation_removals

## Section 2 – Supervised classification

### Stratified per-flow split

Set aside 20% of flows for testing, then set aside 20% of the remaining flows for validation. That gives about 64% training, 16% validation, and 20% testing overall. Both steps stratify by `label`, so each subset keeps roughly the same class proportions as the full dataset. For example, `ddos_ntp` makes up about 1.5% of all flows and should make up about 1.5% of each subset. The fixed random seed makes the split repeatable.

Use the original measurements in `X` for the split. When training a classifier, fit a fresh copy of the preprocessing pipeline on the training rows so that feature filtering, scaling, and PCA learn from those rows alone. Use validation to compare model choices and save the test set for the final evaluation.


In [ ]:
random_seed = 42
test_fraction = 0.20
validation_fraction_of_remaining = 0.20

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=test_fraction, stratify=y, random_state=random_seed
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=validation_fraction_of_remaining,
    stratify=y_train_val,
    random_state=random_seed,
)

In [ ]:
split_counts = pd.DataFrame(
    {
        "All flows": y.value_counts(),
        "Train": y_train.value_counts(),
        "Validation": y_val.value_counts(),
        "Test": y_test.value_counts(),
    }
).sort_index()
split_counts.loc["Total"] = split_counts.sum()
split_counts

### Random forest baseline

Use `clone(preprocess)` as the first step of a new pipeline. The clone copies the preprocessing settings and starts unfitted. When we call `fit` on `X_train`, the filters, scaler, and PCA learn from the training flows. For example, each validation flow is scaled using the training means and standard deviations. The random forest uses its default settings with a fixed seed.

This baseline uses the PCA features from Section 1. A random forest can also learn directly from the measurements. PCA reduces the number of inputs, but each component mixes measurements, which makes individual tree splits harder to interpret.

Compare training and validation accuracy and macro F1. Macro F1 gives each label equal weight, including `ddos_ntp`, which has fewer flows. Save the test set for final evaluation.


In [ ]:
random_forest = Pipeline(
    [
        ("preprocess", clone(preprocess)),
        ("classifier", RandomForestClassifier(random_state=random_seed)),
    ]
)
random_forest.fit(X_train, y_train)

y_train_pred = random_forest.predict(X_train)
y_val_pred = random_forest.predict(X_val)

In [ ]:
scores = pd.DataFrame(
    {
        "Accuracy": [
            accuracy_score(y_train, y_train_pred),
            accuracy_score(y_val, y_val_pred),
        ],
        "Macro F1": [
            f1_score(y_train, y_train_pred, average="macro"),
            f1_score(y_val, y_val_pred, average="macro"),
        ],
    },
    index=["Train", "Validation"],
)
scores.round(4)

In [ ]:
print(classification_report(y_val, y_val_pred, digits=3, zero_division=0))

class_labels = random_forest.classes_
validation_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_val, y_val_pred, labels=class_labels),
    index=pd.Index(class_labels, name="Actual label"),
    columns=pd.Index(class_labels, name="Predicted label"),
)
validation_confusion_matrix

### Tune the number of trees with cross-validation

Use `classifier__n_estimators` to try 100, 200, and 300 trees on the pipeline step named `classifier`. Split `X_train` into five stratified folds so each fold has roughly the same label proportions. For example, the 200-tree pipeline trains on four folds and scores on the fifth, repeating until every fold has been scored once. The filters, scaler, and PCA are fitted again on the training rows of each fold. The separate validation and test sets are reserved for evaluation.

Macro F1 gives every label equal weight, so a rare label such as `ddos_ntp` counts as much as a common one. The search selects the tree count with the highest mean macro F1 and refits that pipeline on all of `X_train`. The table shows the mean score, its standard deviation across folds, and the mean time to fit the full pipeline per fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
forest_search = GridSearchCV(
    random_forest,
    param_grid={"classifier__n_estimators": [100, 200, 300]},
    scoring="f1_macro",
    cv=cv,
)
forest_search.fit(X_train, y_train)

cv_columns = {
    "param_classifier__n_estimators": "Number of trees",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
forest_cv_results = pd.DataFrame(forest_search.cv_results_)[list(cv_columns)].rename(
    columns=cv_columns
)

best_tree_count = forest_search.best_params_["classifier__n_estimators"]
print(f"Selected number of trees: {best_tree_count}")
forest_cv_results.round(4)

### Repeat the search without PCA

Clone the search to reuse the same five stratified folds, tree counts, and macro F1 scoring. Set the PCA and component naming steps to `passthrough` so the forest sees the filtered, standardized measurements. For example, the 200-tree candidate uses the same training and validation rows in both searches. The all-zero filter, Spearman filter, and scaler still learn from the training rows of each fold.

In [ ]:
forest_search_without_pca = clone(forest_search).set_params(
    estimator__preprocess__pca="passthrough",
    estimator__preprocess__name_components="passthrough",
)
forest_search_without_pca.fit(X_train, y_train)

forest_cv_results_without_pca = pd.DataFrame(
    forest_search_without_pca.cv_results_, columns=list(cv_columns)
).rename(columns=cv_columns)

best_tree_count_without_pca = forest_search_without_pca.best_params_[
    "classifier__n_estimators"
]
print(f"Selected number of trees without PCA: {best_tree_count_without_pca}")
forest_cv_results_without_pca.round(4)

### Tune the PCA variance threshold

Fix the random forest at 100 trees, the best tree count in both earlier searches. Compare PCA thresholds of two-thirds, 80%, 90%, and 95% using the same five stratified folds and macro F1 scoring. Each fold fits its own filters, scaler, and PCA using its training rows. For example, at 80%, PCA keeps enough components to explain at least 80% of the variance in that fold's filtered and standardized training data.

In [ ]:
pca_variance_search = clone(forest_search).set_params(
    estimator__classifier__n_estimators=100,
    param_grid={"preprocess__pca__n_components": [2 / 3, 0.80, 0.90, 0.95]},
)
pca_variance_search.fit(X_train, y_train)

pca_cv_columns = {
    "param_preprocess__pca__n_components": "PCA variance threshold",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
pca_cv_results = pd.DataFrame(
    pca_variance_search.cv_results_, columns=list(pca_cv_columns)
).rename(columns=pca_cv_columns)
pca_cv_results["PCA variance threshold"] = pca_cv_results["PCA variance threshold"].map(
    "{:.2%}".format
)

best_variance_threshold = pca_variance_search.best_params_[
    "preprocess__pca__n_components"
]
best_preprocess = pca_variance_search.best_estimator_.named_steps["preprocess"]
best_pca = best_preprocess.named_steps["pca"]
print(f"Selected PCA variance threshold: {best_variance_threshold:.2%}")
print(f"Components kept after refitting on all training rows: {best_pca.n_components_}")
pca_cv_results.round(4)